In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [ ]:
# Sample some text
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# All the characters in the text.
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# Create a mapping from characters to integers and vice versa.
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
import torch

In [ ]:
# Convert all the text into integers and place them in a tensor.
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [ ]:
# Create training and validation sets.
n = round(0.9 * data.shape[0])
train_data = data[:n]
val_data = data[n:]

In [ ]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8 # The number of characters the model will take in for generation. We'll make is larger in a moment.

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')

# Display possible sub-sequences in a batch and the following character we wish to predict.
for i in range(block_size):
    print('input:', xb[0][:i+1], 'target:', yb[0][i].item())

input: tensor([56]) target: 6
input: tensor([56,  6]) target: 0
input: tensor([56,  6,  0]) target: 24
input: tensor([56,  6,  0, 24]) target: 43
input: tensor([56,  6,  0, 24, 43]) target: 58
input: tensor([56,  6,  0, 24, 43, 58]) target: 1
input: tensor([56,  6,  0, 24, 43, 58,  1]) target: 61
input: tensor([56,  6,  0, 24, 43, 58,  1, 61]) target: 46


In [ ]:
block_size = 256

n_embed = 256 # Length of the embedding vector for the characters.
dropout = 0.1
hidden_size = 256 # Size of hidden layers within the RNN

In [ ]:
class RNNModel(torch.nn.Module):
    # Uses a LTSM model
    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, n_embed)
        self.rnn = torch.nn.LSTM(n_embed, hidden_size, batch_first=True)
        self.linear = torch.nn.Linear(hidden_size, vocab_size)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, y=None):
        x = self.embedding(x) # B, T, C
        x, h_n = self.rnn(x)

        if y is not None:
            x = self.dropout(x)
            logits = self.linear(x) # B, T, V
            B, T, V = logits.shape
            logits = logits.view(B * T, V)
            y = y.view(B * T)
            loss = torch.nn.functional.cross_entropy(logits, y)
        else:
            logits = self.linear(x)
            loss = None

        return logits, loss

    def generate(self, x, max_new_tokens):
        # Generates new text

        # x is shape 1, T
        for _ in range(max_new_tokens):
            x_context = x[:, -block_size:]
            logits, _ = self.forward(x_context) # 1, T, V
            probs = torch.softmax(logits[:, -1, :], dim=-1)
            new_char = torch.multinomial(probs, num_samples=1)
            x = torch.concat((x, new_char), dim=1)
        return x  

In [69]:
model = RNNModel()
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')

0.559681 M parameters


In [ ]:
# Without training, the model just generates random characters.
output = model.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)
print(decode(output.tolist()[0]))


T 'pkbHkq
odQjbsEb;Ed-BBxUI3ltTAZnHGx,-QTN'!
?bl!SDBOOAuUukD
gEY:eSUoeKTUSg WL:I&:FolaVkE;jfvBOao-!C


In [ ]:
# For keeping track of training and val losses during training.
eval_iters = 100
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [73]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
# Training
n_batches = 2000
batch_size = 16
eval_interval = 500

for i in range(n_batches):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none = True)
    
    loss.backward()
    optimizer.step()

    # every once in a while evaluate the loss on train and val sets
    if i % eval_interval == 0 or i == n_batches - 1:
        losses = estimate_loss()
        print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

step 0: train loss 1.3019, val loss 1.5222
step 500: train loss 1.2812, val loss 1.5217
step 1000: train loss 1.2645, val loss 1.5046
step 1500: train loss 1.2666, val loss 1.5150
step 1999: train loss 1.2542, val loss 1.5119


In [ ]:
# Sample from our trained model.
output = model.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)
print(decode(output.tolist()[0]))


Of Elgial that the valour'd my redural
Which the villain stull probed the visery my husband
And weigh the violy of the atamious.

LADY ANNE:
If you I shall breathe aid thou to his sorrow:
Since villain, a courtsempet come on possess'd in
The course back again. This farewell move king,
When most not were here dead with the office.
Here, that too so must absence's eye!

LADY CAPULET:
Even sincior! lady for your lady's brother
Woe, they says now wont to me: to have you make thee
To fear him alone.



In [ ]:
# Some more generated text.
output = model.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
print(decode(output.tolist()[0]))


Yoe no place to you, our charity discotry pardon
ropish at instantaties, grants most untair!
We will I besure you on; and here?

First Censpitio, Hers Son:
ganined to the good father-pent souse, give
you eyes disprepent his kindred. The world have
set his ear more sweeth, you go have hears; and then?

Volsce:
Sir, you she hath been cheerful as is it:
A condemn the speaks are, nor wish dead,
And it she find but a hand; and lon and he
This restleasion and no shall be seen
Thy tears cannot perpatiest fair of my
coast and besequed in a part for pubiling gale is.

BUSHY:
I will be beant and brunk, and will unpeopine on?
My lifficing to absence the scare,
That in with all the eviliant you;
I have beaving Henry drunkal assish:
Oo, well; we would have been deton more
his red-sirgh that thy death, with a devil-house.
Ah, whom, makes the poison, in Vienagencd,
Stan my arman prones to surmits, as true,
But happy ears the minesty of the perits
Do seeks his husband name as one kengless;
God no lon

In [ ]:
# It's not perfect, but it can generate real words and use punctuation. It also learns how to place the speaking character's name before
# their dialogue similar to how the original text does. So it definitely learned quite a bit. But the words don't really form coherent thoughts
# or sentences. I think it's not bad for the size of the model / amount of data and the fact that it can only work with characters.